5.1 해리스 특징점 검출 구현

In [1]:
import cv2
import numpy as np

img = np.array([[0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,0],
                [0,0,0,1,0,0,0,0,0,0],
                [0,0,0,1,1,0,0,0,0,0],
                [0,0,0,1,1,1,0,0,0,0],
                [0,0,0,1,1,1,1,0,0,0],
                [0,0,0,1,1,1,1,1,0,0],
                [0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,0]], dtype = np.float32)

ux = np.array([[-1,0,1]])
uy = np.array([-1,0,1]).transpose()
k = cv2.getGaussianKernel(3,1)
g = np.outer(k,k.transpose())

dy = cv2.filter2D(img, cv2.CV_32F, uy)
dx = cv2.filter2D(img, cv2.CV_32F, ux)
dyy = dy*dy
dxx = dx*dx
dyx = dy*dx
gdyy = cv2.filter2D(dyy,cv2.CV_32F,g)
gdxx = cv2.filter2D(dxx,cv2.CV_32F,g)
gdyx = cv2.filter2D(dyx,cv2.CV_32F,g)
C = (gdyy*gdxx - gdyx*gdyx) - 0.04*(gdyy+gdxx)*(gdyy+gdxx)

for j in range(1,C.shape[0]-1):
    for i in range(1,C.shape[1]-1):
        if C[j,i]>0.1 and sum(sum(C[j,i]>C[j-1:j+2,i-1:i+2])) == 8:
            img[j,i] = 9

np.set_printoptions(precision = 2)
print(dy)
print(dx)
print(dyy)
print(dxx)
print(dyx)
print(gdyy)
print(gdxx)
print(gdyx)
print(C)
print(img)

popping = np.zeros([160,160],np.uint8)

for j in range(0,160):
    for i in range(0,160):
        popping[j,i] = np.uint8((C[j//16,i//16]+0.06)*700)

cv2.imshow('imgage display2', popping)
cv2.waitKey()
cv2.destroyAllWindows()

[[ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  1.  1.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  1.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  1.  1.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  1.  1.  0.  0.]
 [ 0.  0.  0. -1. -1. -1. -1.  0.  0.  0.]
 [ 0.  0.  0. -1. -1. -1. -1. -1.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]]
[[ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  1.  0. -1.  0.  0.  0.  0.  0.]
 [ 0.  0.  1.  1. -1. -1.  0.  0.  0.  0.]
 [ 0.  0.  1.  1.  0. -1. -1.  0.  0.  0.]
 [ 0.  0.  1.  1.  0.  0. -1. -1.  0.  0.]
 [ 0.  0.  1.  1.  0.  0.  0. -1. -1.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]]
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 1. 0. 0. 0. 0.]
 [0. 0

SIFT 검출과 기술자 추출

In [1]:
import cv2

img = cv2.imread('bus2.png')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp, des = sift.detectAndCompute(gray,None)

gray = cv2.drawKeypoints(gray,kp,None,flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
cv2.imshow('sift', gray)

cv2.imwrite('buss.jpg', gray)
k = cv2.waitKey()
cv2.destroyAllWindows()

5.5 FLANN 라이브러리를 이용한 sift 매칭

In [26]:
import cv2
import numpy as np
import time

img1 = cv2.imread('bus1.png')[250:400,550:700]
gray1 = cv2.cvtColor(img1,cv2.COLOR_BGR2GRAY)
img2 = cv2.imread('bus2.png')
gray2 = cv2.cvtColor(img2,cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray1,None)
kp2, des2 = sift.detectAndCompute(gray2,None)
print(f'특징점 개수: {len(kp1)}, {len(kp2)}')

start = time.time()
flann_matcher = cv2.DescriptorMatcher_create(cv2.DescriptorMatcher_FLANNBASED)
knn_match = flann_matcher.knnMatch(des1, des2, 2)

T = 0.7
good_match = []

for nearest1, nearest2 in knn_match:
    if (nearest1.distance/nearest2.distance) <T:
        good_match.append(nearest1)
print(f'매칭에 걸린 시간: {time.time() - start}')

img_match = np.empty((max(img1.shape[0], img2.shape[0]), img1.shape[1] + img2.shape[1],3), dtype = np.uint8)
cv2.drawMatches(img1, kp1, img2, kp2, good_match, img_match, flags = cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

cv2.imshow('good matches', img_match)

k = cv2.waitKey()
cv2.destroyAllWindows()

cv2.imwrite('FLANN.jpg', img_match)

특징점 개수: 286, 5989
매칭에 걸린 시간: 0.10975313186645508


True

5.6 RANSAC을 이용해 호모그래피 추정하기

In [15]:
import cv2
import numpy as np

img1 = cv2.imread('bus1.png')[250:400,550:700]
gray1 = cv2.cvtColor(img1,cv2.COLOR_BGR2GRAY)
img2 = cv2.imread('bus2.png')
gray2 = cv2.cvtColor(img2,cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray1,None)
kp2, des2 = sift.detectAndCompute(gray2,None)

flann_matcher = cv2.DescriptorMatcher_create(cv2.DescriptorMatcher_FLANNBASED)
knn_match = flann_matcher.knnMatch(des1, des2, 2)

T = 0.7
good_match = []

for nearest1, nearest2 in knn_match:
    if (nearest1.distance/nearest2.distance) < T:
        good_match.append(nearest1)

points1 = np.float32([kp1[gm.queryIdx].pt for gm in good_match])
points2 = np.float32([kp2[gm.trainIdx].pt for gm in good_match])

H,_ = cv2.findHomography(points1, points2,cv2.RANSAC)

h1, w1 = img1.shape[0], img1.shape[1]
h2, w2 = img2.shape[0], img2.shape[1]

box1 = np.float32([[0,0],[0,h1-1],[w1-1,h1-1],[w1-1,0]]).reshape(4,1,2)
box2 = cv2.perspectiveTransform(box1, H)

img2 = cv2.polylines(img2, [np.int32(box2)], True, (0,255,0),8)

img_match = np.empty((max(h1,h2),w1+w2,3),dtype = np.uint8)
cv2.drawMatches(img1,kp1,img2,kp2,good_match, img_match,flags = cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

cv2.imshow('matches and homography', img_match)

cv2.imwrite('homography.jpg', img_match)

k = cv2.waitKey()
cv2.destroyAllWindows()